In [ ]:
import os

import psycopg2
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

In [ ]:
conexao = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

schemas = pd.read_sql("SELECT schema_name FROM information_schema.schemata ORDER BY schema_name;", conexao)
schemas

In [ ]:
tabelas = pd.read_sql(
    "SELECT table_name, table_type FROM information_schema.tables WHERE table_schema = 'minhaagenda' ORDER BY table_name;",
    conexao,
)
tabelas

## Tabela `minhaagenda.patients`

A tabela já existe (criada pela API) — a célula abaixo só garante isso, com `CREATE TABLE IF NOT EXISTS`.

In [ ]:
cur = conexao.cursor()
cur.execute("""
    CREATE SCHEMA IF NOT EXISTS minhaagenda;

    CREATE TABLE IF NOT EXISTS minhaagenda.patients (
        cpf VARCHAR(11) PRIMARY KEY,
        nome VARCHAR(255) NOT NULL,
        email VARCHAR(255) NOT NULL,
        tel VARCHAR(20) NOT NULL,
        dt_nasc DATE NOT NULL
    );
""")
conexao.commit()
cur.close()

### Create — inserir paciente

In [ ]:
novo_paciente = {
    "nome": "Maria da Silva",
    "cpf": "11144477735",
    "email": "maria.silva@example.com",
    "tel": "11999998888",
    "dt_nasc": "1990-05-20",
}

cur = conexao.cursor()
cur.execute(
    """
    INSERT INTO minhaagenda.patients (nome, cpf, email, tel, dt_nasc)
    VALUES (%(nome)s, %(cpf)s, %(email)s, %(tel)s, %(dt_nasc)s)
    RETURNING cpf;
    """,
    novo_paciente,
)
novo_cpf = cur.fetchone()[0]
conexao.commit()
cur.close()
novo_cpf

### Read — listar / buscar pacientes

In [ ]:
# lista todos os pacientes
pacientes = pd.read_sql("SELECT * FROM minhaagenda.patients ORDER BY cpf;", conexao)
pacientes

In [ ]:
# busca um paciente específico por cpf
paciente = pd.read_sql(
    "SELECT * FROM minhaagenda.patients WHERE cpf = %(cpf)s;",
    conexao,
    params={"cpf": novo_cpf},
)
paciente

### Update — atualizar paciente

In [ ]:
dados_atualizados = {
    "cpf": novo_cpf,
    "tel": "11988887777",
    "email": "maria.silva.novo@example.com",
}

cur = conexao.cursor()
cur.execute(
    """
    UPDATE minhaagenda.patients
    SET tel = %(tel)s,
        email = %(email)s
    WHERE cpf = %(cpf)s;
    """,
    dados_atualizados,
)
conexao.commit()
cur.close()

### Delete — remover paciente

In [ ]:
cur = conexao.cursor()
cur.execute(
    "DELETE FROM minhaagenda.patients WHERE cpf = %(cpf)s;",
    {"cpf": novo_cpf},
)
conexao.commit()
cur.close()